# Entrenamiento modelo predicción de salario

In [101]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/interim/df_merged.csv')
df = df.dropna(subset=['Rango_Salarial'])

In [102]:
df.shape

(19982, 9)

In [103]:
df['Rango_Salarial'].value_counts()

Rango_Salarial
65.000-80.000      3788
<15.000            3435
15.000-22.000      2873
80.000-100.000     2295
30.000-40.000      2027
22.000-30.000      1967
40.000-52.000      1243
100.000-150.000    1177
52.000-65.000       992
>150.000            185
Name: count, dtype: int64

In [104]:
df.head()

,años_experiencia,experiencia,formación_académica,sector,tipo_de_empleo,Ciudad,Salario_medio,Rango_Salarial,id
0,3.0,junior,Grado Universitario,"Salud, Farmacia y Tecnología Médica",Media jornada,Madrid,25000.0,22.000-30.000,0
1,2.0,junior,Ninguna,Servicios,Jornada completa,Madrid,43200.0,40.000-52.000,1
2,1.0,junior,Grado Universitario,Economía y política,Jornada completa,Madrid,13200.0,<15.000,2
3,3.0,senior,Ninguna,Transporte y Logística,NaN,Madrid,131112.0,100.000-150.000,3
4,0.0,intern,Ninguna,Economía y política,Jornada completa,Madrid,15144.0,15.000-22.000,4


## 1. Declarar X e y

In [105]:
X = df.drop(columns=['Rango_Salarial','experiencia']) #me quedo con años en lugar de experiencia porque aporta más información
y = df['Rango_Salarial']
# no borro aun Salario_medio para eliminar outliers

## 2. Dividir en train y test

In [106]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 3. Feature Engineering

### Valores faltantes

In [107]:
X_train.isnull().sum()

años_experiencia        9777
formación_académica        0
sector                     0
tipo_de_empleo         10118
Ciudad                  3129
Salario_medio              0
id                         0
dtype: int64

Por ahora voy a mantener los nulos, ya que modelos como XGBoost o Random Forest pueden trabajar con valores nulos.

### Outliers

La variable salario tiene outliers que pueden afectar al rendimiento del modelo (EDA)

In [108]:
X_train.shape, X_test.shape

((15985, 7), (3997, 7))

In [109]:
IQR = X_train['Salario_medio'].quantile(0.75) - X_train['Salario_medio'].quantile(0.25)
lower = X_train['Salario_medio'].quantile(0.25) - 1.5 * IQR
upper = X_train['Salario_medio'].quantile(0.75) + 1.5 * IQR
X_train = X_train[(X_train['Salario_medio'] >= lower) & (X_train['Salario_medio'] <= upper)]
y_train = y_train[X_train.index]

X_test = X_test[(X_test['Salario_medio'] >= lower) & (X_test['Salario_medio'] <= upper)]
y_test = y_test[X_test.index]

In [110]:
X_train.shape, X_test.shape

((15842, 7), (3961, 7))

In [111]:
# Ya no necesito el salario medio
X_train = X_train.drop(columns=['Salario_medio'])
X_test = X_test.drop(columns=['Salario_medio'])

### Encode variables categoricas

In [112]:
X_train[['formación_académica','sector','tipo_de_empleo','Ciudad']]

,formación_académica,sector,tipo_de_empleo,Ciudad
16721,FP Superior,Tecnología y telecomunicaciones,Temporal,Vigo
23880,Grado Universitario,Deportes y recreación,NaN,Madrid
1222,FP Superior,Servicios,Jornada completa,Murcia
22681,Ninguna,Tecnología y telecomunicaciones,NaN,Madrid
29920,Ninguna,Servicios,NaN,NaN
...,...,...,...,...
23428,Ninguna,Internet,NaN,Zaragoza
17889,Ninguna,Deportes y recreación,Temporal,Palma de Mallorca
28664,Ninguna,Tecnología y telecomunicaciones,NaN,Barcelona
27837,Ninguna,Energía y medio ambiente,NaN,Madrid


In [113]:
pd.get_dummies(X_train['formación_académica'], prefix='formacion', drop_first=True, dummy_na=True).astype(int).head()

,formacion_FP Medio,formacion_FP Superior,formacion_Grado Universitario,formacion_Ninguna,formacion_Postgrado,formacion_nan
16721,0,1,0,0,0,0
23880,0,0,1,0,0,0
1222,0,1,0,0,0,0
22681,0,0,0,1,0,0
29920,0,0,0,1,0,0


In [114]:
pd.get_dummies(X_train['sector'], prefix='sector', drop_first=True).astype(int).head()

,sector_Bienes de consumo,sector_Comercio electrónico,sector_Comercio minorista y comercio,sector_Construcción,sector_Deportes y recreación,sector_Economía y política,sector_Energía y medio ambiente,sector_Finanzas y seguros,sector_Internet,sector_Medios de comunicación,sector_Metales y electrónica,sector_Productos químicos y recursos,sector_Publicidad y Marketing,"sector_Salud, Farmacia y Tecnología Médica",sector_Servicios,sector_Sociedad,sector_Tecnología y telecomunicaciones,sector_Transporte y Logística,"sector_Viajes, turismo y hostelería"
16721,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
23880,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1222,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
22681,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
29920,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0


In [115]:
pd.get_dummies(X_train['tipo_de_empleo'], prefix='tipo_empleo', drop_first=True, dummy_na=True).astype(int).head()

,tipo_empleo_Jornada completa,tipo_empleo_Media jornada,tipo_empleo_Otro,tipo_empleo_Prácticas,tipo_empleo_Temporal,tipo_empleo_nan
16721,0,0,0,0,1,0
23880,0,0,0,0,0,1
1222,1,0,0,0,0,0
22681,0,0,0,0,0,1
29920,0,0,0,0,0,1


In [116]:
X_train = pd.concat([X_train.drop(columns=['formación_académica','sector','tipo_de_empleo','Ciudad']),
                     pd.get_dummies(X_train['formación_académica'], prefix='formacion', drop_first=True, dummy_na=True).astype(int),
                     pd.get_dummies(X_train['sector'], prefix='sector', drop_first=True).astype(int),
                     pd.get_dummies(X_train['tipo_de_empleo'], prefix='tipo_empleo', drop_first=True, dummy_na=True).astype(int)], axis=1)

In [117]:
X_train.head()

,años_experiencia,id,formacion_FP Medio,formacion_FP Superior,formacion_Grado Universitario,formacion_Ninguna,formacion_Postgrado,formacion_nan,sector_Bienes de consumo,sector_Comercio electrónico,...,sector_Sociedad,sector_Tecnología y telecomunicaciones,sector_Transporte y Logística,"sector_Viajes, turismo y hostelería",tipo_empleo_Jornada completa,tipo_empleo_Media jornada,tipo_empleo_Otro,tipo_empleo_Prácticas,tipo_empleo_Temporal,tipo_empleo_nan
16721,0.0,16721,0,1,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0
23880,NaN,23880,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1222,1.0,1222,0,1,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
22681,NaN,22681,0,0,0,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,1
29920,NaN,29920,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [118]:
X_test = pd.concat([X_test.drop(columns=['formación_académica','sector','tipo_de_empleo','Ciudad']),
                     pd.get_dummies(X_test['formación_académica'], prefix='formacion', drop_first=True, dummy_na=True).astype(int),
                     pd.get_dummies(X_test['sector'], prefix='sector', drop_first=True).astype(int),
                     pd.get_dummies(X_test['tipo_de_empleo'], prefix='tipo_empleo', drop_first=True, dummy_na=True).astype(int)], axis=1)

## 4. Feature Scaling

In [119]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train.drop(columns=['id']))
X_test_scaled = scaler.transform(X_test.drop(columns=['id']))

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.drop(columns=['id']).columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.drop(columns=['id']).columns, index=X_test.index)

X_train = pd.concat([X_train_scaled, X_train['id']], axis=1)
X_test = pd.concat([X_test_scaled, X_test['id']], axis=1)

In [120]:
df_embeddings = pd.read_csv('../data/interim/df_embeddings.csv')
X_train = X_train.merge(df_embeddings, on='id', how='left')
X_test = X_test.merge(df_embeddings, on='id', how='left')

X_train = X_train.drop(columns=['id'])
X_test = X_test.drop(columns=['id'])

## 5. Entrenamiento de modelos

In [124]:
sets_posibles = [
    list(X_train.columns),
    list(X_train.drop(columns=['años_experiencia']).columns),
    list(X_train.drop(columns=['años_experiencia','tipo_empleo_Jornada completa','tipo_empleo_Media jornada','tipo_empleo_Otro','tipo_empleo_Prácticas','tipo_empleo_Temporal','tipo_empleo_nan']).columns),
]

In [135]:
import scipy as sp
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
import sklearn.metrics as metrics
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, auc, roc_auc_score

In [ ]:
# Random Forest
rfc = RandomForestClassifier(random_state=42, class_weight='balanced')
param_grid = {"n_estimators": [100, 150],
              'criterion': ['gini', 'entropy', 'log_loss'],
              'max_depth': sp.stats.randint(3, 50),
              'min_samples_split': sp.stats.randint(2, 15),
              'min_samples_leaf': sp.stats.randint(1, 15),
              'max_features': ['sqrt', 'log2', None]
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rfc_model = RandomizedSearchCV(rfc,
                               param_grid,
                               n_iter=25,
                               cv=cv,
                               random_state=42,
                               scoring='f1_macro',
                               n_jobs=-1,
                               verbose=3
                               )


# XGBoost
xgb_model = xgb.XGBClassifier(random_state=42,
                              objective='multi:softprob',
                              eval_metric='mlogloss',
                              num_class=y_train.nunique(),
                              tree_method='hist')

param_grid = {"n_estimators": [100, 150],
              'max_depth': sp.stats.randint(3, 15),
              'learning_rate': sp.stats.uniform(0.01, 0.2),
              'subsample': sp.stats.uniform(0.7, 0.3),
              'colsample_bytree': sp.stats.uniform(0.7, 0.3),
              'min_child_weight': sp.stats.randint(1, 10),
              'gamma': sp.stats.uniform(0, 5),
              'reg_alpha': sp.stats.uniform(0, 1),
              'reg_lambda': sp.stats.uniform(0.5, 2)
}
xgb_model = RandomizedSearchCV(xgb_model,
                               param_grid,
                               n_iter=25,
                               cv=cv,
                               random_state=42,
                               scoring='f1_macro',
                               n_jobs=-1,
                               verbose=3
                               )


# LightGBM
lgb_model = lgb.LGBMClassifier(random_state=42,
                               objective='multiclass',
                               num_class=y_train.nunique(),
                               class_weight='balanced')

param_grid = {"n_estimators": [100, 150],
              'max_depth': sp.stats.randint(3, 15),
              'learning_rate': sp.stats.uniform(0.01, 0.2),
              'num_leaves': sp.stats.randint(15, 80),
              'min_child_samples': sp.stats.randint(5, 50),
              'subsample': sp.stats.uniform(0.7, 0.3),
              'colsample_bytree': sp.stats.uniform(0.7, 0.3),
              'reg_alpha': sp.stats.uniform(0, 1),
              'reg_lambda': sp.stats.uniform(0.5, 2)
}

lgb_model = RandomizedSearchCV(lgb_model,
                               param_grid,
                               n_iter=25,
                               cv=cv,
                               random_state=42,
                               scoring='f1_macro',
                               n_jobs=-1,
                               verbose=3
                               )

In [136]:
from pathlib import Path
import joblib

modelos = [rfc_model,lgb_model,xgb_model]
score_df = pd.DataFrame()
models_dir = Path('../models')
models_dir.mkdir(parents=True, exist_ok=True)

for i, features in enumerate(sets_posibles):
    X_train_fit = X_train[features]
    X_test_fit = X_test[features]
    print(f'Set : {i+1} con {len(features)} features')
    for modelo in modelos:
        print(modelo.estimator.__class__.__name__) #imprimir el nombre del modelo
        modelo.fit(X_train_fit,y_train)
        model_name = modelo.estimator.__class__.__name__
        model_path = models_dir / f'set_{i+1}_{model_name}.joblib'
        joblib.dump(modelo.best_estimator_, model_path)
        # Hacer predicciones
        y_pred_train = modelo.predict(X_train_fit)
        y_pred_proba = modelo.predict_proba(X_test_fit)
        y_pred_test = modelo.predict(X_test_fit)
        
        #Calcular accuracy
        accuracy_test = accuracy_score(y_test,y_pred_test)   
        
        accuracy_train = accuracy_score(y_train,y_pred_train)
        
        #Calcular AUC-ROC
        auc_roc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')
        
        #Crear dataframe
        new_scores = pd.DataFrame({
            "Modelo": [model_name],
            "Feature set": [f'Set : {i+1} con {len(features)} features'],
            "Accuracy test": [accuracy_test],
            "Accuracy train": [accuracy_train],
            "AUC-ROC": [auc_roc]
        })
        score_df = pd.concat([score_df,new_scores],ignore_index=True)

Set : 1 con 203 features
RandomForestClassifier
Fitting 5 folds for each of 25 candidates, totalling 125 fits


KeyboardInterrupt: 